In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from optimizer import run_optimization, load_data

st.set_page_config(page_title="LineWise Optimizer", layout="wide")

st.title("LineWise — Production Recovery Optimizer")
st.write("Upload your equipment status CSV to find the optimal repair strategy.")

uploaded_file = st.file_uploader("Upload Equipment CSV", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.write("### Current Equipment Status")
    st.dataframe(df)

    machines_dict = load_data(df)

    max_possible_cost = sum(
        m.repair_cost for m in machines_dict.values() if m.is_broken
    )

    st.sidebar.header("Maintenance Constraints")
    budget = st.sidebar.slider(
        "Available Budget ($)",
        min_value=0,
        max_value=max(500, int(max_possible_cost)),
        step=500
    )

    best_plan, total_cost, recovered_value = run_optimization(
        machines_dict, budget
    )

    st.write("---")
    st.subheader("Recommended Repair Strategy")

    if not best_plan:
        st.warning(
            "No repair combination restores production within the selected budget."
        )
    else:
        st.success(f"Total Production Value Restored: ${recovered_value:,}")
        st.info(f"Total Repair Cost: ${total_cost:,} (Out of ${budget:,} budget)")

        results_data = [
            {
                "Machine ID": m.machine_id,
                "Repair Cost": m.repair_cost,
                "Value Added": m.production_value
            }
            for m in best_plan
        ]

        results_df = pd.DataFrame(results_data)
        st.table(results_df)